# IO Cloud Agent Cloud — Skills 教学案例

## 从 Raw Tools 到 Skills

| | Notebook 2: Raw Tools | Notebook 3: Skills |
|---|---|---|
| LLM 看到什么 | 18 个原始 MCP 工具 | 3 个高级 Skill |
| 用户说 | "查硬件，然后估价，然后部署" | "部署一个 nginx，最便宜的就行" |
| LLM 调用次数 | 每步都要调一次 LLM | 一次选 Skill，内部自动串联 |
| 可靠性 | 依赖 LLM 每步都选对 | Skill 内部逻辑确定，更稳 |

### 本教程定义 3 个 Skill：

| Skill | 功能 | 内部串联的 MCP 工具 |
|-------|------|------|
| `hardware_scout` | 硬件侦察 | 查硬件目录 → 按价格排序 → 返回推荐 |
| `smart_deploy` | 智能部署 | 查硬件 → 估价 → 部署 → 确认状态 |
| `deployment_manager` | 部署管家 | 列出部署 / 查状态 / 销毁 |

## 0. 环境准备

In [1]:
import os
# os.environ['http_proxy']  = 'http://127.0.0.1:7890'
# os.environ['https_proxy'] = 'http://127.0.0.1:7890'

In [2]:
import json, asyncio, time
from openai import OpenAI
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

# IO Intelligence API (GLM-5.1)
INTELLIGENCE_KEY = 'io-v2-*************'
LLM_BASE_URL = 'https://api.intelligence.io.solutions/api/v1'
LLM_MODEL = 'zai-org/GLM-5.1'

# IO Cloud MCP
CLOUD_KEY = 'io-v2-**********'
MCP_URL = 'https://mcp.io.solutions/mcp'
MCP_HEADERS = {'x-api-key': CLOUD_KEY}

llm = OpenAI(api_key=INTELLIGENCE_KEY, base_url=LLM_BASE_URL)
print('OK')

OK


## 1. MCP 连接层

In [3]:
async def call_mcp_tool(tool_name, arguments=None):
    """MCP 工具调用，带重试。"""
    for attempt in range(3):
        try:
            async with streamablehttp_client(MCP_URL, headers=MCP_HEADERS, timeout=60) as (r, w, _):
                async with ClientSession(r, w) as session:
                    await session.initialize()
                    result = await session.call_tool(tool_name, arguments=arguments or {})
                    text = result.content[0].text if result.content else ''
                    try:
                        return json.loads(text)
                    except json.JSONDecodeError:
                        return text
        except Exception as e:
            if attempt < 2:
                await asyncio.sleep(2)
            else:
                return {'error': str(e)}


def extract_data(resp):
    if isinstance(resp, dict) and 'data' in resp:
        inner = resp['data']
        if isinstance(inner, dict) and 'data' in inner:
            return inner['data']
        return inner
    return resp

print('MCP 连接层 OK')

MCP 连接层 OK


## 2. Skill 框架
每个 Skill 是一个字典，包含：
- `name`: Skill 名称
- `description`: 描述（给 LLM 看的）
- `parameters`: 输入参数 schema
- `handler`: 实际执行函数（内部串联多个 MCP 工具）

In [4]:
# Skill 注册表
SKILLS = {}

def register_skill(name, description, parameters, handler):
    """注册一个 Skill。"""
    SKILLS[name] = {
        'name': name,
        'description': description,
        'parameters': parameters,
        'handler': handler
    }

def skills_to_openai_tools():
    """把所有 Skill 转换为 OpenAI function calling 格式。"""
    return [
        {
            'type': 'function',
            'function': {
                'name': s['name'],
                'description': s['description'],
                'parameters': s['parameters']
            }
        }
        for s in SKILLS.values()
    ]

print('Skill 框架 OK')

Skill 框架 OK


## 3. Skill: hardware_scout (硬件侦察)

一句话查硬件，内部自动：查目录 -> 按价格排序 -> 返回 Top N 推荐。

In [5]:
async def hardware_scout(gpu_filter=None, top_n=5):
    """
    硬件侦察 Skill。
    内部串联: caas_get_hardware_ids -> 排序过滤 -> 返回推荐
    """
    steps = []

    # Step 1: 查硬件目录
    steps.append('[Step 1] 查询 CaaS 硬件目录...')
    raw = await call_mcp_tool('caas_get_hardware_ids')
    items = extract_data(raw)

    # 处理嵌套结构
    if isinstance(items, dict):
        for v in items.values():
            if isinstance(v, list):
                items = v
                break

    if not isinstance(items, list):
        return {'steps': steps, 'error': '无法解析硬件列表', 'raw': str(items)[:500]}

    steps.append(f'  找到 {len(items)} 种硬件')

    # Step 2: 按价格排序
    steps.append('[Step 2] 按价格排序...')
    sorted_items = sorted(items, key=lambda x: x.get('price', 999) if isinstance(x, dict) else 999)

    # Step 3: 可选 GPU 筛选
    if gpu_filter:
        steps.append(f'[Step 3] 筛选 GPU 含 "{gpu_filter}"...')
        sorted_items = [
            it for it in sorted_items
            if isinstance(it, dict) and gpu_filter.lower() in str(it.get('hardware_name', '')).lower()
        ]
        steps.append(f'  筛选后剩 {len(sorted_items)} 种')

    # Step 4: 返回 Top N
    top = sorted_items[:top_n]
    recommendations = []
    for it in top:
        if isinstance(it, dict):
            recommendations.append({
                'hardware_id': it.get('hardware_id'),
                'name': it.get('hardware_name'),
                'price_per_hr': it.get('price'),
                'available': it.get('available'),
                'location': it.get('location'),
                'max_gpus': it.get('max_gpus_per_container')
            })

    steps.append(f'[Done] 返回 Top {len(recommendations)} 推荐')

    return {
        'steps': steps,
        'recommendations': recommendations,
        'total_hardware_count': len(items)
    }


register_skill(
    name='hardware_scout',
    description='硬件侦察: 查询可用 GPU 硬件，按价格排序，返回推荐列表。可按 GPU 型号筛选。',
    parameters={
        'type': 'object',
        'properties': {
            'gpu_filter': {
                'type': 'string',
                'description': 'GPU 型号筛选，如 "H100", "4090", "A100"。不填则返回所有。'
            },
            'top_n': {
                'type': 'integer',
                'description': '返回前 N 个最便宜的，默认 5',
                'default': 5
            }
        }
    },
    handler=hardware_scout
)

print('Skill hardware_scout 已注册')

Skill hardware_scout 已注册


## 4. Skill: smart_deploy (智能部署)

一句话部署，内部自动：查硬件 -> 估价 -> 部署 -> 确认状态。
> 此 Skill 会产生真实费用！

In [6]:
async def smart_deploy(image_url='nginx:latest', duration_hours=1, hardware_id=12, location_ids=None):
    """
    智能部署 Skill。
    内部串联: 估价 -> 部署 -> 查状态
    """
    if location_ids is None:
        location_ids = [2]  # 默认美国

    steps = []

    # Step 1: 估算价格
    steps.append(f'[Step 1] 估算价格: hw_id={hardware_id}, {duration_hours}h...')
    price = await call_mcp_tool('caas_get_price_estimate', {
        'location_ids': location_ids,
        'hardware_id': hardware_id,
        'duration_hours': duration_hours,
        'gpus_per_container': 1,
        'replica_count': 1
    })
    price_data = extract_data(price)
    steps.append(f'  估价结果: {json.dumps(price_data, ensure_ascii=False)[:200]}')

    # Step 2: 部署
    deploy_name = f'skill-deploy-{int(time.time()) % 100000}'
    steps.append(f'[Step 2] 部署容器: {deploy_name}, 镜像={image_url}...')
    deploy_result = await call_mcp_tool('caas_deploy_container', {
        'request': {
            'resource_private_name': deploy_name,
            'duration_hours': duration_hours,
            'gpus_per_container': 1,
            'hardware_id': hardware_id,
            'replica_count': 1,
            'traffic_port': 80,
            'image_url': image_url,
            'location_ids': location_ids
        }
    })

    dep_data = extract_data(deploy_result)
    deployment_id = None
    if isinstance(dep_data, dict):
        deployment_id = dep_data.get('id') or dep_data.get('deployment_id')

    if not deployment_id:
        steps.append(f'  部署可能失败: {json.dumps(deploy_result, ensure_ascii=False)[:300]}')
        return {'steps': steps, 'error': '未获取到 deployment_id', 'raw': deploy_result}

    steps.append(f'  部署成功! ID: {deployment_id}')

    # Step 3: 查询状态
    steps.append('[Step 3] 查询部署状态...')
    status = await call_mcp_tool('caas_get_deployment', {
        'deployment_id': str(deployment_id)
    })
    status_data = extract_data(status)
    steps.append(f'  状态: {status_data.get("status", "unknown") if isinstance(status_data, dict) else "?"}')

    return {
        'steps': steps,
        'deployment_id': deployment_id,
        'deployment_name': deploy_name,
        'status': status_data
    }


register_skill(
    name='smart_deploy',
    description='智能部署: 自动估价并部署容器。会产生真实费用! 默认用 RTX 4090 (hw_id=12) 在美国 (loc=2)。',
    parameters={
        'type': 'object',
        'properties': {
            'image_url': {
                'type': 'string',
                'description': '容器镜像，如 nginx:latest',
                'default': 'nginx:latest'
            },
            'duration_hours': {
                'type': 'integer',
                'description': '部署时长(小时)，默认 1',
                'default': 1
            },
            'hardware_id': {
                'type': 'integer',
                'description': '硬件 ID，默认 12 (RTX 4090)',
                'default': 12
            }
        }
    },
    handler=smart_deploy
)

print('Skill smart_deploy 已注册')

Skill smart_deploy 已注册


## 5. Skill: deployment_manager (部署管家)
一句话管理部署：列出 / 查状态 / 销毁。

In [7]:
async def deployment_manager(action='list', deployment_id=None):
    """
    部署管家 Skill。
    action: list / status / destroy
    """
    steps = []

    if action == 'list':
        steps.append('[Step 1] 列出所有 CaaS 部署...')
        result = await call_mcp_tool('caas_list_deployments', {'page': 1, 'page_size': 10})
        data = extract_data(result)
        deployments = []
        if isinstance(data, dict) and 'deployments' in data:
            deployments = data['deployments']
        steps.append(f'  找到 {len(deployments)} 个部署')
        return {'steps': steps, 'deployments': deployments}

    elif action == 'status' and deployment_id:
        steps.append(f'[Step 1] 查询部署 {deployment_id} 状态...')
        result = await call_mcp_tool('caas_get_deployment', {'deployment_id': deployment_id})
        data = extract_data(result)
        steps.append(f'  状态: {data.get("status", "?") if isinstance(data, dict) else "?"}')

        steps.append('[Step 2] 查询容器详情...')
        containers = await call_mcp_tool('caas_get_deployment_containers', {'deployment_id': deployment_id})
        containers_data = extract_data(containers)

        return {'steps': steps, 'deployment': data, 'containers': containers_data}

    elif action == 'destroy' and deployment_id:
        steps.append(f'[Step 1] 销毁部署 {deployment_id}...')
        result = await call_mcp_tool('caas_destroy_deployment', {'deployment_id': deployment_id})
        steps.append('[Step 2] 确认销毁状态...')
        status = await call_mcp_tool('caas_get_deployment', {'deployment_id': deployment_id})
        status_data = extract_data(status)
        steps.append(f'  状态: {status_data.get("status", "?") if isinstance(status_data, dict) else "?"}')
        return {'steps': steps, 'result': result, 'final_status': status_data}

    else:
        return {'error': f'未知操作: {action}，支持 list/status/destroy'}


register_skill(
    name='deployment_manager',
    description='部署管家: 管理容器部署。action=list 列出所有部署; action=status 查看指定部署状态; action=destroy 销毁指定部署。',
    parameters={
        'type': 'object',
        'properties': {
            'action': {
                'type': 'string',
                'enum': ['list', 'status', 'destroy'],
                'description': '操作类型'
            },
            'deployment_id': {
                'type': 'string',
                'description': '部署 ID，status 和 destroy 时必填'
            }
        },
        'required': ['action']
    },
    handler=deployment_manager
)

print('Skill deployment_manager 已注册')
print(f'\n已注册 {len(SKILLS)} 个 Skills: {list(SKILLS.keys())}')

Skill deployment_manager 已注册

已注册 3 个 Skills: ['hardware_scout', 'smart_deploy', 'deployment_manager']


## 6. Skill 级别的 Agent 循环
LLM 只需要选择哪个 Skill，Skill 内部自动串联多个 MCP 工具。

In [8]:
async def skill_agent_chat(user_message, verbose=True):
    """
    Skill 级别的 Agent 循环:
    用户输入 -> GLM-5.1 选 Skill -> 执行 Skill -> GLM-5.1 总结
    """
    openai_tools = skills_to_openai_tools()

    messages = [
        {'role': 'system', 'content': (
            '你是 IO Cloud GPU 基础设施管理助手。'
            '你有 3 个高级 Skill 可以用，每个 Skill 内部会自动串联多个操作。'
            '请根据用户意图选择合适的 Skill 并填入参数。'
            '用中文回答。'
        )},
        {'role': 'user', 'content': user_message}
    ]

    if verbose:
        print(f'[USER] {user_message}')

    # Round 1: LLM 选择 Skill
    resp = llm.chat.completions.create(
        model=LLM_MODEL,
        messages=messages,
        tools=openai_tools,
        max_tokens=2000
    )

    assistant_msg = resp.choices[0].message
    messages.append(assistant_msg)

    if assistant_msg.tool_calls:
        if verbose and assistant_msg.content:
            print(f'\n[GLM-5.1] {assistant_msg.content}')

        for tc in assistant_msg.tool_calls:
            skill_name = tc.function.name
            skill_args = json.loads(tc.function.arguments) if tc.function.arguments else {}

            if verbose:
                print(f'\n[SKILL] {skill_name}({json.dumps(skill_args, ensure_ascii=False)})')

            # 执行 Skill
            if skill_name in SKILLS:
                handler = SKILLS[skill_name]['handler']
                result = await handler(**skill_args)
            else:
                result = {'error': f'未知 Skill: {skill_name}'}

            if verbose:
                # 打印执行步骤
                if isinstance(result, dict) and 'steps' in result:
                    print()
                    for step in result['steps']:
                        print(f'  {step}')

            messages.append({
                'role': 'tool',
                'tool_call_id': tc.id,
                'content': json.dumps(result, ensure_ascii=False, default=str)
            })

        # Round 2: LLM 总结
        resp2 = llm.chat.completions.create(
            model=LLM_MODEL,
            messages=messages,
            max_tokens=2000
        )
        final_answer = resp2.choices[0].message.content
    else:
        final_answer = assistant_msg.content

    if verbose:
        print(f'\n[ANSWER]\n{final_answer}')

    return final_answer

print('Skill Agent 循环 OK')

Skill Agent 循环 OK


## 7. Demo: 硬件侦察
n一句话触发 `hardware_scout`，内部自动查目录 + 排序 + 筛选。

In [9]:
answer = await skill_agent_chat('帮我看看有哪些 GPU 可以用，最便宜的 5 个是哪些？')

[USER] 帮我看看有哪些 GPU 可以用，最便宜的 5 个是哪些？

[GLM-5.1] 好的，我来帮你查查目前可用的 GPU 硬件，按价格排序返回最便宜的 5 个！

[SKILL] hardware_scout({"gpu_filter": "", "top_n": 5})

  [Step 1] 查询 CaaS 硬件目录...
    找到 59 种硬件
  [Step 2] 按价格排序...
  [Done] 返回 Top 5 推荐

[ANSWER]
查询完成！目前平台共有 **59 种 GPU 硬件**可选，以下是价格最便宜的 **Top 5** 👇

| 排名 | GPU 型号 | 价格 (美元/小时) | 可用数量 | 地区 | 最大 GPU 数 |
|------|----------|-----------------|---------|------|------------|
| 🥇 1 | **RTX A4000** | $0.18 | 14 台 | - | 1 |
| 🥈 2 | **GeForce RTX 3090** | $0.27 | 32 台 | - | 1 |
| 🥉 3 | **GeForce RTX 4090** | $0.30 | 2 台 | - | 1 |
| 4 | **A6000** | $0.43 | 1 台 | 🇺🇸 US | 1 |
| 5 | **T4** | $0.47 | 1 台 | 🇮🇳 IN | 1 |

### 💡 小结：
- **最划算的选择**是 **RTX A4000**，仅 $0.18/小时，且有 14 台可用，性价比很高！
- 如果需要更强的性能，**RTX 3090** 和 **RTX 4090** 价格也很亲民，分别 $0.27 和 $0.30/小时。
- ⚠️ A6000 和 T4 各只有 1 台可用，比较紧缺，如果需要的话建议尽早抢占。

需要我帮你创建一台实例吗？告诉我想用哪个型号就行！🚀


--筛选特定 GPU：

In [10]:
answer = await skill_agent_chat('有没有 H100 可以用？多少钱？')

[USER] 有没有 H100 可以用？多少钱？

[GLM-5.1] 我来帮你查一下 H100 的可用情况和价格！

[SKILL] hardware_scout({"gpu_filter": "H100", "top_n": 5})

  [Step 1] 查询 CaaS 硬件目录...
    找到 59 种硬件
  [Step 2] 按价格排序...
  [Step 3] 筛选 GPU 含 "H100"...
    筛选后剩 8 种
  [Done] 返回 Top 5 推荐

[ANSWER]
好消息，H100 有货！以下是查询到的 **5 款 H100 相关配置**：

| 型号 | 单卡价格 ($/小时) | 当前可用量 | 最大可挂卡数 | 区域 |
|------|------------------|-----------|-------------|------|
| **H100 PCIe** | $1.00 /小时 | 2 张 | 1 | - |
| **H100 80G PCIe** | $1.00 /小时 | **168 张** ✅ | 1 | - |
| **H100** | $1.848 /小时 | 1 张 | 1 | US |
| **H100 80GB HBM3** | $2.09 /小时 | **120 张** ✅ | 1 | - |
| **H100 (SXM5)** | $3.34 /小时 | 1 张 | 1 | CA |

---

### 💡 简要分析：

1. **性价比最高**：**H100 80G PCIe**，仅 **$1.00/小时**，且库存最充足（168 张），非常适合预算敏感的大规模训练任务。
2. **高性能之选**：**H100 80GB HBM3**，使用 HBM3 高带宽显存，适合对显存带宽有更高要求的任务，**$2.09/小时**，库存也很充裕（120 张）。
3. **SXM5 版本**：带宽最高（NVLink），但价格也最贵（$3.34/小时），且目前仅 1 张可用。

> 如果你需要多卡并行（比如 2 卡、4 卡、8 卡），可以告诉我具体需求，我帮你进一步筛选支持多卡的 H100 集群方案！


---\n\n## 8. Demo: 查看我的部署\n\n一句话触发 `deployment_manager`。

In [11]:
answer = await skill_agent_chat('列出我账号下所有的容器部署')

[USER] 列出我账号下所有的容器部署

[GLM-5.1] 好的，我来为您查询所有容器部署信息。

[SKILL] deployment_manager({"action": "list"})

  [Step 1] 列出所有 CaaS 部署...
    找到 10 个部署

[ANSWER]
查询完成！您账号下共有 **10 个容器部署**，详情如下：

| # | 名称 | 部署ID | GPU型号 | 状态 | 完成进度 | 已用计算时间 | 剩余计算时间 |
|---|------|--------|---------|------|----------|-------------|-------------|
| 1 | general - c81841be | f0965932-... | NVIDIA H100 PCIe | ✅ 已完成 | 96.67% | 58 min | 0 min |
| 2 | general - 14cd4fd6 | 89582f79-... | NVIDIA RTX 4090 | ✅ 已完成 | 96.67% | 58 min | 0 min |
| 3 | general - 7d12f953 | 289b935b-... | NVIDIA RTX 4090 | ✅ 已完成 | 96.67% | 58 min | 0 min |
| 4 | general - 68e0ab9f | f1790f6b-... | NVIDIA RTX 4090 | ❌ 已销毁 | 16.67% | 10 min | 0 min |
| 5 | general - 6a78d847 | 2e025f05-... | NVIDIA RTX 4090 | ❌ 已销毁 | 6.67% | 4 min | 0 min |
| 6 | general - 22233c4f | ab1e7d27-... | NVIDIA H100 PCIe | ❌ 已销毁 | 1.67% | 1 min | 0 min |
| 7 | general - 731dd986 | 68e4d5e7-... | NVIDIA RTX 4090 | ❌ 已销毁 | 1.67% | 1 min | 0 min |
| 8 | general - 9d51bddf | 19

---\n\n## 9. Demo: 智能部署 (会花钱!)\n\n> 运行此 Cell 会产生真实费用！完成后请执行下一步销毁。

In [12]:
# 取消注释运行（会花钱）
# answer = await skill_agent_chat('帮我部署一个 nginx，用最便宜的 RTX 4090，1 小时就行')

---\n\n## 10. 交互式对话\n\n修改问题试试：

In [13]:
your_question = '有没有 A100 可以用？多少钱一小时？'

answer = await skill_agent_chat(your_question)

[USER] 有没有 A100 可以用？多少钱一小时？

[SKILL] hardware_scout({"gpu_filter": "A100", "top_n": 5})

  [Step 1] 查询 CaaS 硬件目录...
    找到 59 种硬件
  [Step 2] 按价格排序...
  [Step 3] 筛选 GPU 含 "A100"...
    筛选后剩 6 种
  [Done] 返回 Top 5 推荐

[ANSWER]
有的！目前平台上有多种 A100 GPU 可用，以下是推荐列表：

| 硬件名称 | 价格 (¥/小时) | 可用数量 | 最大 GPU 数 | 地区 |
|---|---|---|---|---|
| **A100-SXM4-80GB** | **1.39** | 17 台 | 1 | — |
| **A100 80GB PCIe** | **1.39** | 5 台 | 1 | — |
| A100 | 2.205 | 1 台 | 1 | 印度 (IN) |
| A100 | 4.1685 | 1 台 | 2 | 印度 (IN) |
| DGX A100 | 7.728 | 1 台 | 8 | 美国 (US) |

### 💡 推荐亮点：
- **性价比最高**：**A100-SXM4-80GB** 和 **A100 80GB PCIe**，仅需 **¥1.39/小时**，且库存充足（分别有 17 台和 5 台可用）。
- **多卡需求**：如果需要多卡并行，可以考虑印度的 2 卡方案或美国的 **DGX A100（8 卡）**，价格为 ¥7.73/小时。

需要我帮你直接开通一台吗？😊


---

## 总结: Raw Tools vs Skills

```
Notebook 2: Raw Tools                    Notebook 3: Skills
────────────────────────────────────────────────────────────────────────────────
用户: "查硬件"                        用户: "部署一个 nginx，最便宜的"
  |                                       |
  v                                       v
LLM -> caas_get_hardware_ids            LLM -> smart_deploy Skill
  |                                       |
  v                                       v
LLM -> "用哪个？"                      Skill 内部自动:
  |                                       1. 估价
  v                                       2. 部署
LLM -> caas_get_price_estimate            3. 查状态
  |                                       |
  v                                       v
LLM -> caas_deploy_container            LLM -> "已部署，花了 $0.30"
  |
  v
LLM -> "部署完成"

4 次 LLM 调用                          2 次 LLM 调用
每步都可能出错                        Skill 内部逻辑确定
```

### 什么时候用 Skills？

- 有固定的多步流程（查硬件 -> 估价 -> 部署）
- 需要更高可靠性（减少 LLM 决策点）
- 想要更简洁的用户体验（一句话搞定）

### 什么时候用 Raw Tools？

- 探索性查询（不确定要调哪个 API）
- 灵活组合（流程不固定）
- 开发调试阶段